# MVP — Previsão de Custo de Garantia (Mês+1, Mês+2, Mês+3)

**Nome:** _Seu nome aqui_  
**Matrícula:** _Sua matrícula aqui_  
**Data:** _dd/mm/aaaa_  
**Dataset:** Ordens de Serviço de Garantia — Imbera Brasil (2022–2025)  
**Tipo de problema:** Séries Temporais / Forecasting com features de lag (ML-based)  

---

## Checklist do MVP

| Item | Status |
|---|---|
| Problema definido com contexto, objetivo e tipo de tarefa | ☐ |
| Dataset descrito, com fonte, atributos e restrições | ☐ |
| Dataset carregado por URL pública ou fonte diretamente acessível | ☐ |
| Análise exploratória objetiva, conectada à modelagem | ☐ |
| Divisão temporal adequada (sem embaralhamento) | ☐ |
| Prevenção de vazamento de dados | ☐ |
| Features de lag e calendário criadas e justificadas | ☐ |
| Modelo baseline definido (média móvel) | ☐ |
| Pelo menos dois modelos comparados | ☐ |
| Avaliação com MAE e MAPE | ☐ |
| Discussão de limitações e melhorias | ☐ |
| Código limpo, organizado e executável do início ao fim | ☐ |
| Conclusão conectada ao objetivo inicial | ☐ |

---
# 1. Definição do Problema

## 1.1 Descrição do problema

A Imbera Brasil fabrica congeladores e freezers comerciais (modelos EVZ21, EVF19, entre outros) que são distribuídos para clientes dos segmentos Foodservice, KOF e outros canais. Após a venda, os equipamentos entram em período de garantia, gerando ordens de serviço (OS) para atendimento técnico — com custos de peças, mão de obra e despesas adicionais.

O volume e o custo dessas OS variam ao longo do tempo, impactando diretamente o planejamento financeiro da área de pós-venda. Hoje, a empresa não possui um modelo preditivo para antecipar esses gastos, o que dificulta o provisionamento de recursos e a gestão de fornecedores de assistência técnica.

**Usuários da solução:** área financeira e de pós-venda da Imbera Brasil.  
**Relevância:** antecipar os custos de garantia permite melhor alocação de budget, negociação com prestadores e redução de surpresas financeiras.

## 1.2 Objetivo do MVP

> O objetivo deste MVP é construir e avaliar modelos de Machine Learning para **prever o custo total de garantia dos meses seguintes (mês+1, mês+2 e mês+3)** a partir do histórico mensal de ordens de serviço, comparando uma abordagem baseline (média móvel) com modelos candidatos baseados em features de lag e calendário, e discutindo suas limitações.

**Objetivo deste trabalho:**  
> _Preencha aqui com suas palavras após entender bem o problema._

## 1.3 Tipo de problema

**Tipo escolhido:** Séries Temporais / Forecasting (Regressão com features temporais)  
**Justificativa:** O target (`custo_total_mes`) é uma variável numérica contínua que evolui ao longo do tempo, com possível sazonalidade mensal e tendência. A ordem temporal dos dados é fundamental — embaralhar os registros causaria vazamento de dados (data leakage). Por isso, adotamos a abordagem de ML com features de lag, que transforma o problema de série temporal em um problema de regressão supervisionada preservando a ordem cronológica.

## 1.4 Premissas, hipóteses e critérios de sucesso

**Hipóteses iniciais:**
1. O custo de garantia apresenta sazonalidade mensal (ex.: picos em determinados meses do ano).
2. O custo dos meses anteriores (lags) é o principal preditor do custo futuro.
3. O volume de OS por mês está correlacionado com o custo total.

**Critérios de sucesso:**
- **Métrica principal:** MAPE (Mean Absolute Percentage Error) — intuitivo para comunicar ao negócio.
- **Métrica secundária:** MAE (Mean Absolute Error) — em reais, para dimensionar o erro absoluto.
- **Resultado mínimo esperado:** superar o baseline (média móvel) em pelo menos 10% no MAPE.
- **Restrição prática:** modelo simples o suficiente para ser re-treinado mensalmente com baixo custo computacional.

---
# 2. Ambiente, Bibliotecas e Reprodutibilidade

In [ ]:
# === Setup básico e reprodutibilidade ===
import os
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print('Python:', sys.version.split()[0])
print('Pandas:', pd.__version__)
print('Seed:', SEED)

## 2.1 Funções auxiliares

In [ ]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — evita divisão por zero."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def evaluate_forecast(y_true, y_pred, model_name='modelo'):
    """Calcula MAE, RMSE, R2 e MAPE para regressão/forecasting."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mp   = mape(y_true, y_pred)
    return {'Modelo': model_name, 'MAE (R$)': round(mae, 2),
            'RMSE (R$)': round(rmse, 2), 'R2': round(r2, 4),
            'MAPE (%)': round(mp, 2)}


def plot_forecast(df_monthly, y_pred_dict, horizon_label):
    """Plota série real vs previsões para um horizonte."""
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(df_monthly['ano_mes'], df_monthly['custo_total'],
            marker='o', label='Real', linewidth=2)
    colors = ['tomato', 'steelblue', 'seagreen', 'orange']
    for (name, preds), color in zip(y_pred_dict.items(), colors):
        ax.plot(df_monthly['ano_mes'].iloc[-len(preds):],
                preds, marker='s', linestyle='--',
                label=name, color=color)
    ax.set_title(f'Custo Real vs Previsto — {horizon_label}')
    ax.set_xlabel('Mês')
    ax.set_ylabel('Custo Total (R$)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'R$ {x:,.0f}'))
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


print('Funções auxiliares carregadas.')

---
# 3. Carga dos Dados

## 3.1 Fonte dos dados

- **Dataset:** Ordens de Serviço de Garantia — Imbera Brasil
- **Período:** Janeiro/2022 a Junho/2025 (aproximadamente)
- **Volume:** ~50.000+ registros
- **Fonte:** Sistema interno de gestão de pós-venda da Imbera
- **Restrições:** Dados confidenciais — não publicar em repositórios abertos. Para execução no Colab, carregar via Google Drive ou upload direto.
- **Privacidade:** Contém nome de prestadores de serviço e clientes — anonimizar se necessário para apresentação.

> **TODO:** Substituir o bloco abaixo pelo carregamento real da sua base.

In [ ]:
# ============================================================
# OPÇÃO 1 — Carregar do Google Drive (recomendado para Colab)
# ============================================================
# from google.colab import drive
# drive.mount('/content/drive')
# caminho = '/content/drive/MyDrive/SEU_ARQUIVO.xlsx'   # <-- ajuste o caminho
# df_raw = pd.read_excel(caminho)

# ============================================================
# OPÇÃO 2 — Upload direto no Colab
# ============================================================
# from google.colab import files
# uploaded = files.upload()          # abre janela para selecionar o arquivo
# import io
# df_raw = pd.read_excel(io.BytesIO(list(uploaded.values())[0]))

# ============================================================
# OPÇÃO 3 — URL pública (ex.: GitHub ou Google Sheets exportado)
# ============================================================
# url = 'https://...SEU_LINK_PUBLICO...'   # <-- substitua
# df_raw = pd.read_csv(url, sep=';', decimal=',')

# ============================================================
# MODO EXEMPLO — dados sintéticos para o notebook rodar
# Remova este bloco quando usar sua base real!
# ============================================================
USE_EXAMPLE_DATA = True   # TODO: altere para False após configurar sua base

if USE_EXAMPLE_DATA:
    np.random.seed(SEED)
    n = 55000
    datas = pd.date_range('2022-01-01', periods=42, freq='MS')
    meses = np.random.choice(datas, size=n)
    sazonalidade = 1 + 0.3 * np.sin(2 * np.pi * pd.DatetimeIndex(meses).month / 12)
    tendencia = 1 + 0.002 * (pd.DatetimeIndex(meses).year - 2022) * 12
    custo_base = np.random.lognormal(mean=4.8, sigma=0.9, size=n)
    df_raw = pd.DataFrame({
        'OS': range(n),
        'Mês Fechamento - KOF': meses,
        'FAMÍLIA DO PRODUTO': np.random.choice(['CONGELADOR', 'REFRIGERADOR'], size=n, p=[0.7, 0.3]),
        'Produto ajustado': np.random.choice(['EVZ21', 'EVF19', 'EVZ31'], size=n),
        'Classificação cliente': np.random.choice(['FOODSERVICE', 'KOF', 'TERCEIROS'], size=n),
        'Cliente ajustado': np.random.choice(['IMBERA - GARANTIA', 'GARANTIA', 'KOF'], size=n),
        'Defeito Constatado': np.random.choice([
            'Controlador com defeito', 'Termostato com defeito',
            'Fonte de led queimado', 'Porta com defeito',
            'Filtro secador obstruído', 'Vazamento de Gás Refrigerante',
            'Motor evaporador com defeito', 'Condensador com vazamento'
        ], size=n),
        'Gastos com peças': np.random.exponential(scale=50, size=n).round(2),
        'MÃO DE OBRA': np.random.choice([0, 70, 100, 110], size=n),
        'Valor Adicional': np.random.choice([0, 0, 0, 35, 110], size=n),
        'SOMA DE GASTOS': (custo_base * sazonalidade * tendencia).round(2),
        'Ano': pd.DatetimeIndex(meses).year,
        'Mês': pd.DatetimeIndex(meses).month,
    })
    print('Modo EXEMPLO ativo. Base sintética gerada com', len(df_raw), 'linhas.')
    print('Substitua pelo carregamento real para o MVP final.')
else:
    print('Base real carregada com', len(df_raw), 'linhas.')

df_raw.head(3)

## 3.2 Visão geral do dataset

In [ ]:
print('Formato:', df_raw.shape)
print('\nTipos de dados:')
display(df_raw.dtypes.to_frame('tipo'))
print('\nValores ausentes:')
display(df_raw.isna().sum().to_frame('ausentes'))
print('\nDuplicatas:', df_raw.duplicated().sum())

## 3.3 Dicionário de dados

| Coluna | Tipo | Descrição | Usada no modelo? |
|---|---|---|---|
| OS | ID | Número da ordem de serviço | Não — identificador |
| Mês Fechamento - KOF | Data | Mês de encerramento da OS | Sim — base temporal |
| FAMÍLIA DO PRODUTO | Categórico | Tipo do produto (Congelador, Refrigerador) | Não na série agregada |
| Produto ajustado | Categórico | Modelo do equipamento (EVZ21, EVF19...) | Não na série agregada |
| Defeito Constatado | Texto | Tipo de defeito relatado | Não na série agregada |
| Classificação cliente | Categórico | Segmento do cliente | Não na série agregada |
| SOMA DE GASTOS | Numérico | Custo total da OS (peças + MO + adicional) | **TARGET** (após agregar por mês) |
| Gastos com peças | Numérico | Custo de peças da OS | Componente do target |
| MÃO DE OBRA | Numérico | Custo de mão de obra da OS | Componente do target |
| Valor Adicional | Numérico | Despesas adicionais | Componente do target |

---
# 4. Análise Exploratória dos Dados (EDA)

In [ ]:
# === Padronizar coluna de data e limpar valores monetários ===
# Ajuste o nome da coluna de data conforme sua base real
DATE_COL = 'Mês Fechamento - KOF'
COST_COL = 'SOMA DE GASTOS'

df = df_raw.copy()
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')

# Se a coluna de custo for string com 'R$', descomente e ajuste:
# df[COST_COL] = (df[COST_COL].astype(str)
#                 .str.replace('R\$', '', regex=True)
#                 .str.replace('.', '', regex=False)
#                 .str.replace(',', '.', regex=False)
#                 .str.strip()
#                 .astype(float))

df = df.dropna(subset=[DATE_COL, COST_COL])
df = df[df[COST_COL] >= 0]   # remover valores negativos inválidos

print('Período:', df[DATE_COL].min().strftime('%b/%Y'), 'até', df[DATE_COL].max().strftime('%b/%Y'))
print('Registros válidos:', len(df))

In [ ]:
# === Agregar por mês ===
df['ano_mes'] = df[DATE_COL].dt.to_period('M')

df_monthly = (
    df.groupby('ano_mes')
    .agg(
        custo_total=(COST_COL, 'sum'),
        volume_os=(COST_COL, 'count'),
        ticket_medio=(COST_COL, 'mean'),
        custo_pecas=('Gastos com peças', 'sum') if 'Gastos com peças' in df.columns else (COST_COL, 'sum'),
        custo_mo=('MÃO DE OBRA', 'sum') if 'MÃO DE OBRA' in df.columns else (COST_COL, 'sum'),
    )
    .reset_index()
    .sort_values('ano_mes')
)

df_monthly['ano_mes_dt'] = df_monthly['ano_mes'].dt.to_timestamp()

print('Meses disponíveis:', len(df_monthly))
display(df_monthly.head(6))

In [ ]:
# === Gráfico 1: Série histórica de custo total ===
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_monthly['ano_mes_dt'], df_monthly['custo_total'],
        marker='o', linewidth=2, color='steelblue')
ax.set_title('Custo Total de Garantia por Mês')
ax.set_xlabel('Mês')
ax.set_ylabel('Custo Total (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"Custo médio mensal: R$ {df_monthly['custo_total'].mean():,.2f}")
print(f"Custo mínimo:       R$ {df_monthly['custo_total'].min():,.2f}")
print(f"Custo máximo:       R$ {df_monthly['custo_total'].max():,.2f}")

In [ ]:
# === Gráfico 2: Sazonalidade mensal ===
df_monthly['mes_num'] = df_monthly['ano_mes'].dt.month
sazon = df_monthly.groupby('mes_num')['custo_total'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(sazon.index, sazon.values, color='steelblue', edgecolor='white')
ax.set_title('Custo Médio por Mês do Ano (Sazonalidade)')
ax.set_xlabel('Mês')
ax.set_ylabel('Custo Médio (R$)')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Jan','Fev','Mar','Abr','Mai','Jun',
                     'Jul','Ago','Set','Out','Nov','Dez'])
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# === Gráfico 3: Volume de OS vs Custo Total ===
fig, ax1 = plt.subplots(figsize=(14, 4))
ax2 = ax1.twinx()
ax1.bar(df_monthly['ano_mes_dt'], df_monthly['volume_os'],
        alpha=0.4, color='gray', label='Volume de OS')
ax2.plot(df_monthly['ano_mes_dt'], df_monthly['custo_total'],
         color='steelblue', marker='o', linewidth=2, label='Custo Total')
ax1.set_ylabel('Volume de OS')
ax2.set_ylabel('Custo Total (R$)')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
ax1.set_title('Volume de OS vs Custo Total por Mês')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4.1 Síntese da análise exploratória

**Preencha após observar os gráficos acima:**

- A série apresenta tendência de crescimento, estabilidade ou queda?
- Há sazonalidade visível? Em quais meses o custo tende a ser maior?
- Volume de OS e custo total andam juntos? Ou o ticket médio varia bastante?
- Há outliers ou meses atípicos que merecem atenção?

**Síntese:**  
> _Preencha aqui após analisar os gráficos com sua base real._

---
# 5. Engenharia de Features e Divisão Temporal

In [ ]:
# === Criação de features de lag e calendário ===
# Estratégia: transformar o problema de série temporal em regressão supervisionada.
# Cada linha representa um mês; as features são os custos dos meses anteriores (lags)
# e indicadores de calendário (mês, trimestre).

df_feat = df_monthly[['ano_mes', 'ano_mes_dt', 'custo_total', 'volume_os', 'ticket_medio']].copy()

# Features de lag — custo dos meses anteriores
for lag in [1, 2, 3, 6, 12]:
    df_feat[f'lag_{lag}'] = df_feat['custo_total'].shift(lag)

# Médias móveis — suavizam ruído e capturam tendência
df_feat['mm3']  = df_feat['custo_total'].shift(1).rolling(window=3).mean()
df_feat['mm6']  = df_feat['custo_total'].shift(1).rolling(window=6).mean()
df_feat['mm12'] = df_feat['custo_total'].shift(1).rolling(window=12).mean()

# Features de calendário
df_feat['mes_num']    = df_feat['ano_mes'].dt.month
df_feat['trimestre']  = df_feat['ano_mes'].dt.quarter
df_feat['ano']        = df_feat['ano_mes'].dt.year

# Variação percentual mês a mês (momentum)
df_feat['variacao_pct_1'] = df_feat['custo_total'].pct_change(1).shift(1)
df_feat['variacao_pct_3'] = df_feat['custo_total'].pct_change(3).shift(1)

# Volume de OS do mês anterior (preditor adicional)
df_feat['volume_lag1'] = df_feat['volume_os'].shift(1)
df_feat['volume_lag2'] = df_feat['volume_os'].shift(2)

# Remover linhas com NaN geradas pelos lags
df_feat = df_feat.dropna().reset_index(drop=True)

print('Meses disponíveis após criação de features:', len(df_feat))
display(df_feat.head(3))

In [ ]:
# === Criação dos targets para mês+1, mês+2 e mês+3 ===
# Abordagem: Direct Multi-Step — um modelo separado para cada horizonte.
# É mais simples e evita acumulação de erros da abordagem recursiva.

df_feat['target_h1'] = df_feat['custo_total'].shift(-1)  # mês+1
df_feat['target_h2'] = df_feat['custo_total'].shift(-2)  # mês+2
df_feat['target_h3'] = df_feat['custo_total'].shift(-3)  # mês+3

FEATURE_COLS = [
    'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12',
    'mm3', 'mm6', 'mm12',
    'mes_num', 'trimestre', 'ano',
    'variacao_pct_1', 'variacao_pct_3',
    'volume_lag1', 'volume_lag2'
]

print('Features usadas no modelo:', FEATURE_COLS)
print('Targets: target_h1 (mês+1), target_h2 (mês+2), target_h3 (mês+3)')

In [ ]:
# === Divisão temporal treino / teste ===
# IMPORTANTE: nunca embaralhar dados de série temporal!
# Usamos os últimos 6 meses como teste (holdout temporal).

N_TEST = 6   # últimos N meses como conjunto de teste

# Para cada horizonte, descartamos as últimas linhas sem target válido
def split_horizon(df, target_col, n_test):
    df_h = df.dropna(subset=[target_col]).copy()
    train = df_h.iloc[:-n_test]
    test  = df_h.iloc[-n_test:]
    return train, test

train_h1, test_h1 = split_horizon(df_feat, 'target_h1', N_TEST)
train_h2, test_h2 = split_horizon(df_feat, 'target_h2', N_TEST)
train_h3, test_h3 = split_horizon(df_feat, 'target_h3', N_TEST)

print(f'Horizonte +1 — Treino: {len(train_h1)} meses | Teste: {len(test_h1)} meses')
print(f'Horizonte +2 — Treino: {len(train_h2)} meses | Teste: {len(test_h2)} meses')
print(f'Horizonte +3 — Treino: {len(train_h3)} meses | Teste: {len(test_h3)} meses')
print(f'\nPeríodo de teste: {test_h1["ano_mes"].min()} a {test_h1["ano_mes"].max()}')

## 5.1 Justificativa da divisão

A divisão é estritamente temporal: os **últimos 6 meses** formam o conjunto de teste e todos os meses anteriores compõem o treino. Isso simula o cenário real de uso — o modelo é treinado com dados históricos e avaliado em meses que ele "nunca viu".

**Por que não embaralhar?** Em séries temporais, embaralhar causa vazamento de dados (data leakage): o modelo poderia "aprender" com o futuro, gerando métricas artificialmente boas que não se repetem em produção.

**Por que 3 modelos separados (Direct Multi-Step)?** É mais simples e robusto do que a abordagem recursiva (onde o erro do mês+1 alimenta a previsão do mês+2). Cada modelo é especializado em seu horizonte.

---
# 6. Baseline

In [ ]:
# === Baseline: Média Móvel dos últimos 3 meses ===
# Interpretação: "o próximo mês vai custar o mesmo que a média dos 3 últimos meses".
# É o benchmark mínimo — qualquer modelo deve superá-lo para justificar sua complexidade.

results_all = []

for horizon, (train, test) in enumerate([
    (train_h1, test_h1),
    (train_h2, test_h2),
    (train_h3, test_h3)
], start=1):
    target_col = f'target_h{horizon}'
    y_test = test[target_col].values
    y_baseline = test['mm3'].values   # média móvel de 3 meses já está no dataset
    res = evaluate_forecast(y_test, y_baseline, model_name=f'Baseline MM3 (H+{horizon})')
    results_all.append(res)

pd.DataFrame(results_all)

---
# 7. Modelos Candidatos

## 7.1 Justificativa dos modelos

| Modelo | Justificativa |
|---|---|
| **Ridge Regression** | Linear, rápido, interpretável. Bom ponto de partida para séries sem muita não-linearidade. A regularização L2 evita overfitting com poucas amostras mensais. |
| **Random Forest Regressor** | Captura não-linearidades e interações entre features (ex.: sazonalidade + lag combinados). Robusto a outliers. |
| **Gradient Boosting Regressor** | Geralmente o modelo com melhor performance em dados tabulares. Mais sensível a hiperparâmetros, mas pode superar os anteriores com ajuste adequado. |

In [ ]:
# === Treinamento e avaliação — todos os modelos, todos os horizontes ===

models = {
    'Ridge':            Ridge(alpha=1.0),
    'RandomForest':     RandomForestRegressor(n_estimators=100, random_state=SEED),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=SEED),
}

trained = {}   # armazena os modelos treinados por (modelo, horizonte)
results_models = []

for horizon, (train, test) in enumerate([
    (train_h1, test_h1),
    (train_h2, test_h2),
    (train_h3, test_h3)
], start=1):
    target_col = f'target_h{horizon}'
    X_train = train[FEATURE_COLS]
    y_train = train[target_col]
    X_test  = test[FEATURE_COLS]
    y_test  = test[target_col].values

    for name, model in models.items():
        t0 = time.time()
        model.fit(X_train, y_train)
        elapsed = round(time.time() - t0, 3)
        y_pred = model.predict(X_test)
        res = evaluate_forecast(y_test, y_pred, model_name=f'{name} (H+{horizon})')
        res['Tempo (s)'] = elapsed
        results_models.append(res)
        trained[(name, horizon)] = (model, test, y_pred, target_col)

df_results = pd.DataFrame(results_models)
display(df_results.sort_values(['Modelo']))

In [ ]:
# === Tabela resumo comparativa por horizonte ===
df_compare = pd.concat([
    pd.DataFrame(results_all),
    df_results
]).reset_index(drop=True)

print('=== MAPE (%) — menor é melhor ===')
pivot = df_compare.pivot_table(index='Modelo', values='MAPE (%)')
display(pivot.sort_values('MAPE (%)'))

## 7.2 Análise dos resultados iniciais

**Preencha após observar a tabela acima:**

- Algum modelo superou o baseline de forma consistente?
- O MAPE aumenta conforme o horizonte (H+1 → H+3)? Isso é esperado?
- O Random Forest ou Gradient Boosting se saíram melhor que o Ridge? Por quê isso faz sentido?

**Resposta:**  
> _Preencha aqui._

---
# 8. Visualização das Previsões

In [ ]:
# === Plotar real vs previsto para cada horizonte ===

for horizon in [1, 2, 3]:
    preds_dict = {}
    test_ref = None
    for name in models:
        model, test, y_pred, target_col = trained[(name, horizon)]
        preds_dict[name] = y_pred
        test_ref = test

    # Baseline
    preds_dict['Baseline MM3'] = test_ref['mm3'].values

    # Plot
    fig, ax = plt.subplots(figsize=(12, 4))
    y_real = test_ref[f'target_h{horizon}'].values
    x_axis = test_ref['ano_mes_dt'].values
    ax.plot(x_axis, y_real, marker='o', linewidth=2, label='Real', color='black')
    colors = ['tomato', 'steelblue', 'seagreen', 'orange']
    for (nm, pred), color in zip(preds_dict.items(), colors):
        ax.plot(x_axis, pred, marker='s', linestyle='--', label=nm, color=color)
    ax.set_title(f'Real vs Previsto — Horizonte Mês+{horizon}')
    ax.set_xlabel('Mês')
    ax.set_ylabel('Custo Total (R$)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

---
# 9. Otimização de Hiperparâmetros (Modelo Escolhido)

In [ ]:
# === Otimização do GradientBoosting para o horizonte H+1 ===
# Usamos TimeSeriesSplit para preservar a ordem temporal na validação cruzada.
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

tscv = TimeSeriesSplit(n_splits=5)

param_dist = {
    'n_estimators':    randint(50, 300),
    'max_depth':       randint(2, 8),
    'learning_rate':   uniform(0.01, 0.2),
    'min_samples_split': randint(2, 10),
    'subsample':       uniform(0.6, 0.4)
}

gb_base = GradientBoostingRegressor(random_state=SEED)

search = RandomizedSearchCV(
    gb_base,
    param_distributions=param_dist,
    n_iter=20,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    random_state=SEED,
    n_jobs=1,
    verbose=1
)

X_train_h1 = train_h1[FEATURE_COLS]
y_train_h1 = train_h1['target_h1']

search.fit(X_train_h1, y_train_h1)

print('\nMelhor MAE na validação: R$', round(-search.best_score_, 2))
print('Melhores hiperparâmetros:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

## 9.1 Discussão da otimização

**Perguntas para responder:**

- A otimização melhorou o MAPE em relação ao modelo inicial?
- O TimeSeriesSplit é adequado aqui? Por que não usar KFold tradicional?
- Quais hiperparâmetros tiveram mais impacto?

**Resposta:**  
> _Preencha aqui._

---
# 10. Avaliação Final

In [ ]:
# === Avaliar modelo otimizado no conjunto de teste (H+1) ===
best_model_h1 = search.best_estimator_
X_test_h1 = test_h1[FEATURE_COLS]
y_test_h1 = test_h1['target_h1'].values

y_pred_best = best_model_h1.predict(X_test_h1)
res_best = evaluate_forecast(y_test_h1, y_pred_best, 'GB Otimizado (H+1)')

print('=== Resultado Final — Modelo Otimizado (Horizonte Mês+1) ===')
display(pd.DataFrame([res_best]))

# Comparar com baseline
res_bl = results_all[0]  # baseline H+1
mape_melhora = res_bl['MAPE (%)'] - res_best['MAPE (%)']
print(f"\nMelhora sobre o baseline: {mape_melhora:.2f} p.p. de MAPE")
if mape_melhora > 0:
    print('✅ Modelo superou o baseline.')
else:
    print('⚠️  Modelo não superou o baseline — revisar features ou tentar mais dados.')

In [ ]:
# === Importância das features ===
feat_imp = pd.Series(
    best_model_h1.feature_importances_,
    index=FEATURE_COLS
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Importância das Features — GradientBoosting (H+1)')
ax.set_xlabel('Importância')
plt.tight_layout()
plt.show()

In [ ]:
# === Resíduos ===
residuos = y_test_h1 - y_pred_best

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.scatter(y_pred_best, residuos, color='steelblue', alpha=0.7)
ax1.axhline(0, linestyle='--', color='red')
ax1.set_title('Resíduos vs Valores Previstos')
ax1.set_xlabel('Previsto (R$)')
ax1.set_ylabel('Resíduo (R$)')

ax2.hist(residuos, bins=15, color='steelblue', edgecolor='white')
ax2.set_title('Distribuição dos Resíduos')
ax2.set_xlabel('Resíduo (R$)')
plt.tight_layout()
plt.show()

## 10.1 Análise de erros e limitações

**Preencha após analisar os gráficos:**

- Os resíduos estão distribuídos de forma aleatória ou há padrão sistemático?
- O modelo erra mais nos meses de maior ou menor custo?
- Há sinais de overfitting (desempenho muito melhor no treino do que no teste)?
- Em quais situações o modelo não deveria ser usado?
- O que poderia melhorar o modelo (mais dados, mais features, outro algoritmo)?

**Resposta:**  
> _Preencha aqui._

---
# 11. Comparação Final dos Modelos

| Modelo | Horizonte | MAPE (%) | MAE (R$) | Observações |
|---|---|---:|---:|---|
| Baseline MM3 | H+1 | _preencha_ | _preencha_ | Referência mínima |
| Ridge | H+1 | _preencha_ | _preencha_ | Linear, rápido |
| Random Forest | H+1 | _preencha_ | _preencha_ | Não-linear |
| GradientBoosting | H+1 | _preencha_ | _preencha_ | Melhor candidato |
| GB Otimizado | H+1 | _preencha_ | _preencha_ | **Modelo final** |
| GB Otimizado | H+2 | _preencha_ | _preencha_ | Horizonte 2 |
| GB Otimizado | H+3 | _preencha_ | _preencha_ | Horizonte 3 |

> **Comentário:** preencha com os valores reais após executar o notebook com sua base.

---
# 12. Boas Práticas e Rastreabilidade

| Decisão | Justificativa | Impacto esperado |
|---|---|---|
| Seed 42 | Reprodutibilidade | Resultados idênticos em re-execuções |
| Divisão temporal (sem shuffle) | Evitar data leakage | Avaliação realista |
| Direct Multi-Step (3 modelos) | Evitar acumulação de erro | Previsões independentes por horizonte |
| TimeSeriesSplit na validação cruzada | Preservar ordem temporal no tuning | Hiperparâmetros sem vazamento |
| MAPE como métrica principal | Interpretável para o negócio (%) | Facilita comunicação com stakeholders |
| Features de lag 1, 2, 3, 6, 12 | Capturar tendência e sazonalidade anual | Melhora previsões de longo prazo |
| Média móvel como baseline | Benchmark simples e realista | Define o mínimo aceitável de desempenho |

---
# 13. Conclusão

**Preencha após executar o notebook com sua base real:**

- **Objetivo:** prever o custo total de garantia para os meses +1, +2 e +3 a partir do histórico de ordens de serviço.
- **Melhor solução encontrada:** _descreva o modelo e as métricas finais._
- **Comparação com baseline:** _o modelo superou a média móvel? Em quanto?_
- **Principais aprendizados:** _o que os dados ensinaram sobre o comportamento dos custos de garantia?_
- **Limitações:** _poucos meses de histórico? Dados faltantes? Eventos externos não capturados?_
- **Próximos passos:** _incluir features por modelo de produto, criar dashboard de monitoramento, re-treinar mensalmente._

**Conclusão:**  
> _Preencha aqui._

---
# 14. Salvamento de Artefatos (Opcional)

In [ ]:
# Descomente para salvar o modelo final e os resultados

# import joblib
# joblib.dump(best_model_h1, 'modelo_garantia_h1.pkl')
# print('Modelo H+1 salvo.')

# df_compare.to_csv('resultados_modelos.csv', index=False)
# print('Tabela de resultados salva.')